# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hussainhhgh/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
month_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

feature_query = f"""
    SELECT content_hash_id, client_hash_id,
        SUM(gsc_impressions) as impressions_15d,
        SUM(gsc_clicks) as clicks_15d,
        AVG(gsc_avg_position) as avg_position_15d
    FROM read_parquet('{month_path}')
    WHERE report_date <= DATE '2026-03-15' AND gsc_data_available = TRUE
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) > 0
"""
features_df = con.sql(feature_query).df()

label_query = f"""
    SELECT content_hash_id, SUM(gsc_clicks) as clicks_1631
    FROM read_parquet('{month_path}')
    WHERE report_date >= DATE '2026-03-16' AND gsc_data_available = TRUE
    GROUP BY content_hash_id
"""
labels_df = con.sql(label_query).df()

merged = features_df.merge(labels_df, on='content_hash_id', how='left')
merged['clicks_1631'] = merged['clicks_1631'].fillna(0)
merged['ctr_15d'] = merged['clicks_15d'] / merged['impressions_15d']
merged['declining'] = (merged['clicks_1631'] < merged['clicks_15d'] * 0.9).astype(int)

def position_bucket(pos):
    if pos <= 3: return 'top_3'
    elif pos <= 10: return 'page_1'
    elif pos <= 20: return 'striking'
    elif pos <= 50: return 'page_3_5'
    else: return 'deep'
merged['position_tier'] = merged['avg_position_15d'].apply(position_bucket)

print(f"Rows rebuilt: {len(merged)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows rebuilt: 151981


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

**Distributions:** Key numeric fields show severe right-skew in raw form — impressions_15d (12.89), clicks_15d (57.82), and ctr_15d (21.14) — driven by a small number of very high-traffic pages against a long tail of low-traffic ones. Applying a log1p transform confirms these tails are real, not artifacts: log_impressions_15d skew drops to 0.00, and log_clicks_15d improves from 57.82 to 1.96 (still somewhat skewed due to the large share of zero-click pages, which log1p compresses but doesn't fully normalize). avg_position_15d and ctr_15d are left in their raw, bounded form since they're already interpretable without transformation. This confirms mean-based summaries on raw impressions/clicks would be misleading; log-transformed or bucketed views are more reliable, and log_impressions_15d/log_clicks_15d are worth carrying forward as candidate features for the modeling notebook.

In [10]:
import numpy as np

print("Skew before log1p transform:")
for col in ['impressions_15d', 'clicks_15d', 'ctr_15d']:
    print(f"  {col}: {merged[col].skew():.2f}")

merged['log_impressions_15d'] = np.log1p(merged['impressions_15d'])
merged['log_clicks_15d'] = np.log1p(merged['clicks_15d'])

print("\nSkew after log1p transform:")
print(f"  log_impressions_15d: {merged['log_impressions_15d'].skew():.2f}")
print(f"  log_clicks_15d: {merged['log_clicks_15d'].skew():.2f}")

Skew before log1p transform:
  impressions_15d: 12.89
  clicks_15d: 57.82
  ctr_15d: 21.14

Skew after log1p transform:
  log_impressions_15d: 0.00
  log_clicks_15d: 1.96


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

**Signal test verdicts:**

**Signal 1 (position_tier → CTR-fix/refresh logic): OPPOSITE.** Decline risk is lowest for deep pages (2.6%) and rises through the visible tiers, peaking at top_3 (22.4%) and page_1 (21.3%) — the reverse of the naive expectation that worse positions decline more. This matches the earlier finding in w04_baseline_score.ipynb.

**Signal 2 (ctr_15d → CTR-fix logic): CONFIRMED, strongly.** Zero-CTR pages show 0% decline by construction (a page with zero clicks in the first half cannot show a >10% click drop from a nonzero baseline, so this reflects the label definition rather than an independent CTR effect). Excluding that mechanical artifact, "active" CTR pages (58.5%) decline meaningfully more than "low" CTR pages (44.2%) — a real, if partially confounded, signal.

**Signal 3 (impressions_15d → volume/quick-win logic): CONFIRMED, cleanly.** Decline risk rises monotonically and substantially with impression volume — from 2.6% (low) to 7.9% (mid_low) to 22.9% (mid_high) to 41.4% (high). This is the strongest, cleanest signal audited so far — high-traffic pages are far more likely to see a second-half click drop, plausibly because they have more room to regress from an unusually strong first half, or because they're more exposed to ranking volatility.

In [11]:
print("=== Signal 1: position_tier ===")
s1 = merged.groupby('position_tier').agg(n=('content_hash_id','count'), pct_declining=('declining','mean'))
print(s1)

print("\n=== Signal 2: ctr_tier ===")
merged['ctr_tier'] = merged['ctr_15d'].apply(lambda x: 'zero' if x==0 else ('low' if x<0.002 else 'active'))
s2 = merged.groupby('ctr_tier').agg(n=('content_hash_id','count'), pct_declining=('declining','mean'))
print(s2)

print("\n=== Signal 3: impressions_tier (volume) ===")
merged['impressions_tier'] = pd.qcut(merged['impressions_15d'], q=4, labels=['low','mid_low','mid_high','high'], duplicates='drop')
s3 = merged.groupby('impressions_tier', observed=True).agg(n=('content_hash_id','count'), pct_declining=('declining','mean'))
print(s3)

=== Signal 1: position_tier ===
                   n  pct_declining
position_tier                      
deep            9770       0.025691
page_1         70000       0.212943
page_3_5       27741       0.158682
striking       26913       0.180991
top_3          17557       0.223842

=== Signal 2: ctr_tier ===
               n  pct_declining
ctr_tier                       
active     37913       0.584892
low        13993       0.442007
zero      100075       0.000000

=== Signal 3: impressions_tier (volume) ===
                      n  pct_declining
impressions_tier                      
low               38712       0.026142
mid_low           37418       0.079507
mid_high          37886       0.228950
high              37965       0.413512


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**Flag-linked test — CTR-fix flag assumption: REVERSED.** The assumption tested was that high-impression, low/zero-CTR pages (a fixable snippet/title problem) would show elevated decline risk. The data shows the opposite: this segment declines *less* often (14.3%, n=43,141) than all other pages (20.4%, n=108,840). A plausible explanation is that pages with high impressions but near-zero CTR are often already at their performance floor — they can't decline much further in relative terms if they were barely converting clicks to begin with, similar to the deep-position pattern in Signal 1. This is a genuinely useful negative result: it suggests the CTR-fix flag identifies pages worth *improving* for click capture, but is not by itself a good predictor of near-term decline risk — those are two different problems that shouldn't be conflated in a single rule.

In [12]:
print("=== Flag-linked test: CTR-fix flag assumption ===")
print("Assumption: high impressions + low/zero CTR = fixable snippet/title problem")

high_imp_low_ctr = merged[(merged['impressions_15d'] > merged['impressions_15d'].median()) & (merged['ctr_15d'] < 0.002)]
rest = merged[~merged.index.isin(high_imp_low_ctr.index)]

print(f"\nHigh-impression + low-CTR pages: n={len(high_imp_low_ctr)}, pct_declining={high_imp_low_ctr['declining'].mean():.3f}")
print(f"All other pages: n={len(rest)}, pct_declining={rest['declining'].mean():.3f}")

=== Flag-linked test: CTR-fix flag assumption ===
Assumption: high impressions + low/zero CTR = fixable snippet/title problem

High-impression + low-CTR pages: n=43141, pct_declining=0.143
All other pages: n=108840, pct_declining=0.204


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

**What this means in practice:** A content team should not use the CTR-fix flag as a decline predictor — it tests a different, valid problem (click capture), but this audit shows it doesn't correlate with near-term decline risk the way it might be assumed to. The strongest, most trustworthy decline signal found across this audit is impression volume: high-traffic pages should be prioritized for decline monitoring specifically because they showed a clean, monotonic risk increase (2.6% → 41.4% across volume tiers), unlike the noisier or reversed patterns seen in position and CTR-based signals.

In [13]:
print("Practical takeaway for a content team:")
print("""
1. High-impression, low-CTR pages are NOT a reliable decline-risk signal on their own —
   they actually decline slightly less than average. The CTR-fix flag should be treated
   as a separate lever (click-capture optimization) from decline-risk triage, not merged into one rule.
2. Impression volume (Signal 3) is the strongest, cleanest decline predictor found in this audit —
   high-traffic pages are the ones most likely to see a real drop, likely due to more room to regress.
3. Position risk is concentrated in visible, ranked pages (top_3/page_1), not deep/unranked pages
   — consistent with earlier weeks' findings on this same reversal.
""")

Practical takeaway for a content team:

1. High-impression, low-CTR pages are NOT a reliable decline-risk signal on their own — 
   they actually decline slightly less than average. The CTR-fix flag should be treated 
   as a separate lever (click-capture optimization) from decline-risk triage, not merged into one rule.
2. Impression volume (Signal 3) is the strongest, cleanest decline predictor found in this audit — 
   high-traffic pages are the ones most likely to see a real drop, likely due to more room to regress.
3. Position risk is concentrated in visible, ranked pages (top_3/page_1), not deep/unranked pages 
   — consistent with earlier weeks' findings on this same reversal.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.